# Integrated CA1 – Neural Networks on Apache Spark with a Parquet data lake
**MSc in Data Analytics – Advanced Data Analytics / Big Data Storage and Processing**

This notebook is the coded experiment that accompanies the paper. It covers:
1. **Storage** – CSV versus columnar Parquet at three data scales.
2. **Data preparation and iterative neural-network design** – every iteration is logged with validation metrics.
3. **Final evaluation** on a held-out test set, compared with single-node scikit-learn.
4. **Scalability** – training time versus data volume, and the effect of partitioning.
5. **Streaming inference** with Spark Structured Streaming.

All numbers quoted in the paper are written to `results.json` by this notebook.

In [ ]:
import os, time, json, shutil, warnings, urllib.request
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, Bucketizer
from pyspark.ml.classification import LogisticRegression, MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
warnings.filterwarnings("ignore")

QUICK = os.environ.get("QUICK", "0") == "1"      # set QUICK=1 for a fast smoke test
SEED = 42
WORK = os.path.abspath("lake")                   # local stand-in for an object store / data lake
os.makedirs(WORK, exist_ok=True); os.makedirs("figures", exist_ok=True)
RESULTS = {}

spark = (SparkSession.builder.master("local[*]").appName("CA1-NN-on-Spark")
         .config("spark.driver.memory", "2g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.enabled", "false").config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
RESULTS["env"] = {"spark": spark.version, "cores": os.cpu_count(), "python_pkg": "pyspark"}
print(RESULTS["env"])

## 1. Dataset
The *diamonds* data set (53,940 rows, 10 columns; Wickham, 2016) is loaded from a public GitHub mirror.
Task: classify a stone into one of four **price bands** (quartiles of price) from its physical attributes.
`price` itself is never used as a feature.

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"
if not os.path.exists("diamonds.csv"):
    urllib.request.urlretrieve(URL, "diamonds.csv")

schema = StructType([StructField("carat", DoubleType()), StructField("cut", StringType()),
                     StructField("color", StringType()), StructField("clarity", StringType()),
                     StructField("depth", DoubleType()), StructField("table", DoubleType()),
                     StructField("price", DoubleType()), StructField("x", DoubleType()),
                     StructField("y", DoubleType()), StructField("z", DoubleType())])
raw = spark.read.csv("diamonds.csv", header=True, schema=schema)
n_raw = raw.count()
print("rows:", n_raw)
raw.describe(["carat", "depth", "table", "price", "x", "y", "z"]).show()

## 2. Storage experiment – CSV versus Parquet
The real file is small, so larger volumes are synthesised by replicating rows with small random jitter on
the numeric columns (so that Parquet compression is not inflated by exact duplicates). This is used **only** for
storage and timing benchmarks, never for accuracy figures.

In [ ]:
def inflate(df, k):
    """Replicate df k times with +/-1% multiplicative jitter on numeric columns."""
    if k == 1:
        return df
    reps = df.withColumn("rep", F.explode(F.array(*[F.lit(i) for i in range(k)])))
    for c in ["carat", "depth", "table", "price", "x", "y", "z"]:
        reps = reps.withColumn(c, F.col(c) * (1 + (F.rand(SEED) - 0.5) * 0.02))
    return reps.drop("rep")

def dir_size_mb(path):
    tot = 0
    for r, _, fs in os.walk(path):
        for f in fs:
            if not f.startswith(".") and not f.endswith(".crc"):
                tot += os.path.getsize(os.path.join(r, f))
    return tot / 1e6

def timed(fn, reps=3):
    ts = []
    for _ in range(reps):
        t = time.time(); fn(); ts.append(time.time() - t)
    return float(np.median(ts))

scales = [1, 5, 20] if QUICK else [1, 20, 100]
storage_rows = []
for k in scales:
    d = inflate(raw, k)
    p_csv, p_pq = f"{WORK}/diamonds_x{k}_csv", f"{WORK}/diamonds_x{k}_parquet"
    for p in (p_csv, p_pq):
        shutil.rmtree(p, ignore_errors=True)
    t = time.time(); d.write.option("header", True).csv(p_csv); w_csv = time.time() - t
    t = time.time(); d.write.option("compression", "snappy").parquet(p_pq); w_pq = time.time() - t
    n = spark.read.parquet(p_pq).count()
    q_csv = timed(lambda: spark.read.csv(p_csv, header=True, schema=schema).groupBy("cut").agg(F.avg("price")).collect())
    q_pq = timed(lambda: spark.read.parquet(p_pq).groupBy("cut").agg(F.avg("price")).collect())
    storage_rows.append(dict(scale=k, rows=n, csv_mb=dir_size_mb(p_csv), parquet_mb=dir_size_mb(p_pq),
                             write_csv_s=w_csv, write_parquet_s=w_pq, query_csv_s=q_csv, query_parquet_s=q_pq))
    print(storage_rows[-1])
storage = pd.DataFrame(storage_rows)
storage["size_ratio"] = storage.csv_mb / storage.parquet_mb
storage["query_speedup"] = storage.query_csv_s / storage.query_parquet_s
RESULTS["storage"] = storage.to_dict(orient="records")
storage.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(3.45, 1.9))
x = np.arange(len(storage)); w = 0.38
lab = [f"{r/1e6:.2f}M" if r >= 1e6 else f"{r/1e3:.0f}k" for r in storage.rows]
ax[0].bar(x - w/2, storage.csv_mb, w, label="CSV", color="#888888")
ax[0].bar(x + w/2, storage.parquet_mb, w, label="Parquet", color="#1f4e79")
ax[0].set_xticks(x); ax[0].set_xticklabels(lab, fontsize=6); ax[0].set_ylabel("Size (MB)", fontsize=7)
ax[0].set_yscale("log"); ax[0].tick_params(labelsize=6); ax[0].legend(fontsize=6, frameon=False)
ax[0].set_title("(a) On-disk size", fontsize=7)
ax[1].bar(x - w/2, storage.query_csv_s, w, color="#888888"); ax[1].bar(x + w/2, storage.query_parquet_s, w, color="#1f4e79")
ax[1].set_xticks(x); ax[1].set_xticklabels(lab, fontsize=6); ax[1].set_ylabel("Query time (s)", fontsize=7)
ax[1].tick_params(labelsize=6); ax[1].set_title("(b) Group-by query", fontsize=7)
for a in ax: a.set_xlabel("Rows", fontsize=7)
plt.tight_layout(); plt.savefig("figures/fig_storage.png", dpi=300); plt.close()

In [3]:
clean = raw.filter((F.col("x") > 0) & (F.col("y") > 0) & (F.col("z") > 0) & (F.col("y") < 20) & (F.col("z") < 10))
n_clean = clean.count()
print("removed rows:", n_raw - n_clean)
RESULTS["data"] = {"rows_raw": n_raw, "rows_clean": n_clean, "removed": n_raw - n_clean}

train_all_raw, test_raw = clean.randomSplit([0.8, 0.2], seed=SEED)
edges = train_all_raw.approxQuantile("price", [0.25, 0.5, 0.75], 0.0005)
print("price-band edges (from training data):", edges)
RESULTS["data"]["edges"] = edges
bucket = Bucketizer(splits=[-float("inf")] + edges + [float("inf")], inputCol="price", outputCol="label")
train_all, test = bucket.transform(train_all_raw), bucket.transform(test_raw)
train, val = train_all.randomSplit([0.8, 0.2], seed=SEED)
for nm, d in [("train", train), ("val", val), ("test", test)]:
    print(nm, d.count())
RESULTS["data"].update({"train": train.count(), "val": val.count(), "test": test.count(), "train_all": train_all.count()})
print(train.groupBy("label").count().orderBy("label").toPandas().T)

NameError: name 'raw' is not defined

In [4]:
NUM_RAW = ["carat", "depth", "table", "x", "y", "z"]
CATS = ["cut", "color", "clarity"]
acc_ev = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
f1_ev = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

def add_eng(df):
    return (df.withColumn("log_carat", F.log("carat"))
              .withColumn("log_volume", F.log(F.col("x") * F.col("y") * F.col("z")))
              .withColumn("aspect", F.col("x") / F.col("y")))

def feature_stages(use_cat, use_eng, scale):
    stages, cols = [], list(NUM_RAW)
    if use_eng:
        cols += ["log_carat", "log_volume", "aspect"]
    if use_cat:
        stages += [StringIndexer(inputCols=CATS, outputCols=[c + "_i" for c in CATS], handleInvalid="keep"),
                   OneHotEncoder(inputCols=[c + "_i" for c in CATS], outputCols=[c + "_oh" for c in CATS])]
        cols += [c + "_oh" for c in CATS]
    stages.append(VectorAssembler(inputCols=cols, outputCol="raw_features", handleInvalid="keep"))
    if scale:
        stages.append(StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True))
    else:
        stages.append(VectorAssembler(inputCols=["raw_features"], outputCol="features"))
    return stages

LOG = []
def run(name, use_cat, use_eng, scale, hidden=None, max_iter=100, solver="l-bfgs", step=0.03, note=""):
    tr, va = add_eng(train), add_eng(val)
    fp = Pipeline(stages=feature_stages(use_cat, use_eng, scale)).fit(tr)
    tr_f, va_f = fp.transform(tr).select("features", "label").cache(), fp.transform(va).select("features", "label").cache()
    d = len(tr_f.first()["features"])
    if hidden is None:
        est = LogisticRegression(maxIter=max_iter, family="multinomial", labelCol="label", featuresCol="features")
        layers = None
    else:
        layers = [d] + hidden + [4]
        est = MultilayerPerceptronClassifier(layers=layers, maxIter=max_iter, solver=solver, stepSize=step,
                                             seed=SEED, blockSize=128, labelCol="label", featuresCol="features")
    t = time.time(); m = est.fit(tr_f); fit_s = time.time() - t
    pv = m.transform(va_f)
    row = dict(run=name, model="LogReg" if hidden is None else "MLP", cat=use_cat, eng=use_eng, scaled=scale,
               layers=str(layers) if layers else "-", solver=solver if hidden else "l-bfgs", iters=max_iter,
               n_features=d, val_acc=acc_ev.evaluate(pv), val_f1=f1_ev.evaluate(pv), fit_s=fit_s, note=note)
    LOG.append(row); tr_f.unpersist(); va_f.unpersist()
    print(f"{name:4s} {row['model']:6s} feats={d:2d} layers={row['layers']:<18s} val_acc={row['val_acc']:.4f} f1={row['val_f1']:.4f} fit={fit_s:5.1f}s  {note}")
    return row

maj = train.groupBy("label").count().orderBy(F.desc("count")).first()["label"]
maj_acc = val.filter(F.col("label") == maj).count() / val.count()
LOG.append(dict(run="R0", model="Majority", cat=False, eng=False, scaled=False, layers="-", solver="-", iters=0,
                n_features=0, val_acc=maj_acc, val_f1=np.nan, fit_s=0.0, note="floor"))
print(f"R0   majority class val_acc={maj_acc:.4f}")

# --- Iteration block A: baselines and data preparation -------------------------------------------------------
run("R1", False, False, False, None, 100, note="LogReg, raw numeric")
run("R2", False, False, False, [16], 100, note="MLP on unscaled features")
run("R3", False, False, True,  [16], 100, note="+ standardisation")
run("R4", True,  False, True,  [16], 100, note="+ one-hot cut/color/clarity")
run("R5", True,  True,  True,  [16], 100, note="+ log/volume/aspect features")

NameError: name 'MulticlassClassificationEvaluator' is not defined

In [5]:
# --- Iteration block B: architecture and optimiser (using the R5 feature set) -------------------------------
run("R6", True, True, True, [32], 100, note="wider")
run("R7", True, True, True, [64, 32], 100, note="deeper")
if not QUICK:
    run("R8", True, True, True, [128, 64, 32], 100, note="deeper + wider")
run("R9", True, True, True, [64, 32], 100, solver="gd", step=0.03, note="mini-batch gradient descent (untuned step)")

NameError: name 'run' is not defined

In [6]:
log = pd.DataFrame(LOG)
log.to_csv("experiment_log.csv", index=False)
RESULTS["log"] = json.loads(log.to_json(orient="records"))
print(log[["run", "model", "n_features", "layers", "solver", "iters", "val_acc", "val_f1", "fit_s", "note"]].round(4).to_string(index=False))

NameError: name 'pd' is not defined

In [7]:
mlp_runs = log[log.model == "MLP"].sort_values(["val_acc", "fit_s"], ascending=[False, True])
best = mlp_runs.iloc[0]
print("\nselected on validation:", best.run, best.layers, "iters", best.iters, best.solver, "val_acc", round(best.val_acc, 4))
BEST = dict(hidden=json.loads(best.layers)[1:-1], iters=int(best.iters), solver=best.solver)
RESULTS["best"] = dict(run=best.run, **BEST, val_acc=float(best.val_acc))

fig, ax = plt.subplots(figsize=(3.45, 2.0))
cols = ["#888888" if m in ("Majority", "LogReg") else ("#1f4e79" if r == best.run else "#5b9bd5") for r, m in zip(log.run, log.model)]
ax.bar(log.run, log.val_acc * 100, color=cols)
ax.set_ylim(min(log.val_acc) * 100 - 5, 100)
ax.set_ylabel("Validation accuracy (%)", fontsize=7); ax.tick_params(labelsize=6)
ax.set_xlabel("Experiment run (R0 = majority baseline)", fontsize=7)
for i, v in enumerate(log.val_acc * 100):
    ax.text(i, v + 0.3, f"{v:.1f}", ha="center", fontsize=5)
plt.tight_layout(); plt.savefig("figures/fig_iterations.png", dpi=300); plt.close()

NameError: name 'log' is not defined

In [8]:
final_stages = feature_stages(True, True, True)
d_final = None
tr_all_e, te_e = add_eng(train_all), add_eng(test)
fp = Pipeline(stages=final_stages).fit(tr_all_e)
d_final = len(fp.transform(tr_all_e).first()["features"])
layers = [d_final] + BEST["hidden"] + [4]
mlp = MultilayerPerceptronClassifier(layers=layers, maxIter=BEST["iters"], solver=BEST["solver"], seed=SEED,
                                     blockSize=128, labelCol="label", featuresCol="features")
full_pipe = Pipeline(stages=final_stages + [mlp])
t = time.time(); full_model = full_pipe.fit(tr_all_e); final_fit_s = time.time() - t
pt = full_model.transform(te_e)
test_acc, test_f1 = acc_ev.evaluate(pt), f1_ev.evaluate(pt)
print(f"FINAL MLP {layers}: test accuracy={test_acc:.4f}  weighted F1={test_f1:.4f}  fit={final_fit_s:.1f}s")

lr_pipe = Pipeline(stages=final_stages + [LogisticRegression(maxIter=100, family="multinomial", labelCol="label")])
t = time.time(); lr_model = lr_pipe.fit(tr_all_e); lr_fit_s = time.time() - t
plr = lr_model.transform(te_e)
lr_acc, lr_f1 = acc_ev.evaluate(plr), f1_ev.evaluate(plr)
print(f"LogReg on same features: test accuracy={lr_acc:.4f}  weighted F1={lr_f1:.4f}  fit={lr_fit_s:.1f}s")

cm = (pt.groupBy("label", "prediction").count().toPandas().pivot(index="label", columns="prediction", values="count").fillna(0).astype(int))
print(cm)
RESULTS["final"] = dict(layers=layers, test_acc=test_acc, test_f1=test_f1, fit_s=final_fit_s,
                        lr_test_acc=lr_acc, lr_test_f1=lr_f1, lr_fit_s=lr_fit_s, confusion=cm.values.tolist())
full_model.write().overwrite().save(f"{WORK}/models/final_pipeline")

NameError: name 'feature_stages' is not defined

In [9]:
from sklearn.neural_network import MLPClassifier
tr_feat = fp.transform(tr_all_e).select(vector_to_array("features").alias("f"), "label").toPandas()
te_feat = fp.transform(te_e).select(vector_to_array("features").alias("f"), "label").toPandas()
Xtr, ytr = np.vstack(tr_feat.f.values), tr_feat.label.values
Xte, yte = np.vstack(te_feat.f.values), te_feat.label.values
sk = MLPClassifier(hidden_layer_sizes=tuple(BEST["hidden"]), max_iter=200, random_state=SEED, early_stopping=True)
t = time.time(); sk.fit(Xtr, ytr); sk_fit_s = time.time() - t
sk_acc = float((sk.predict(Xte) == yte).mean())
print(f"scikit-learn MLP: test accuracy={sk_acc:.4f}  fit={sk_fit_s:.1f}s")
RESULTS["sklearn"] = dict(test_acc=sk_acc, fit_s=sk_fit_s)

NameError: name 'fp' is not defined

In [ ]:
base = fp.transform(tr_all_e).select("features", "label")
d_dim = d_final
ks = [1, 5] if QUICK else [1, 5, 10]
scal = []
for k in ks:
    rep = base if k == 1 else base.withColumn("r", F.explode(F.array(*[F.lit(i) for i in range(k)]))).drop("r")
    rep = rep.repartition(4).cache(); n = rep.count()
    est = MultilayerPerceptronClassifier(layers=[d_dim] + BEST["hidden"] + [4], maxIter=20, seed=SEED, blockSize=128)
    t = time.time(); est.fit(rep); s_fit = time.time() - t
    row = dict(k=k, rows=n, spark_fit_s=s_fit, sk_fit_s=np.nan)
    if k <= 10:
        Xk, yk = np.tile(Xtr, (k, 1)), np.tile(ytr, k)
        skm = MLPClassifier(hidden_layer_sizes=tuple(BEST["hidden"]), max_iter=20, random_state=SEED, tol=0, n_iter_no_change=100)
        t = time.time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore"); skm.fit(Xk, yk)
        row["sk_fit_s"] = time.time() - t
    scal.append(row); print(row); rep.unpersist()
scal = pd.DataFrame(scal)
RESULTS["scalability"] = json.loads(scal.to_json(orient="records"))

fig, ax = plt.subplots(figsize=(3.45, 2.0))
ax.plot(scal.rows / 1e6, scal.spark_fit_s, "o-", color="#1f4e79", label="Spark MLlib MLP", markersize=3)
ax.plot(scal.rows / 1e6, scal.sk_fit_s, "s--", color="#c55a11", label="scikit-learn MLP", markersize=3)
ax.set_xlabel("Training rows (millions)", fontsize=7); ax.set_ylabel("Training time (s)", fontsize=7)
ax.tick_params(labelsize=6); ax.legend(fontsize=6, frameon=False); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("figures/fig_scalability.png", dpi=300); plt.close()

In [ ]:
# Partitioning overhead on the same machine
k_part = 3
rep = base.withColumn("r", F.explode(F.array(*[F.lit(i) for i in range(k_part)]))).drop("r")
part_rows = []
for p in [1, 4, 16]:
    r_ = rep.repartition(p).cache(); n = r_.count()
    est = MultilayerPerceptronClassifier(layers=[d_dim] + BEST["hidden"] + [4], maxIter=20, seed=SEED, blockSize=128)
    t = time.time(); est.fit(r_); part_rows.append(dict(partitions=p, rows=n, fit_s=time.time() - t)); r_.unpersist()
    print(part_rows[-1])
RESULTS["partitions"] = part_rows

In [ ]:
from pyspark.ml import PipelineModel
stream_dir, ck = f"{WORK}/stream_in", f"{WORK}/ckpt"
shutil.rmtree(stream_dir, ignore_errors=True); shutil.rmtree(ck, ignore_errors=True)
s_k = 5 if QUICK else 20
stream_src = inflate(test_raw, s_k)
stream_src = bucket.transform(stream_src)
stream_src.repartition(20).write.parquet(stream_dir)
n_stream = spark.read.parquet(stream_dir).count()
model = PipelineModel.load(f"{WORK}/models/final_pipeline")
sdf = spark.readStream.schema(spark.read.parquet(stream_dir).schema).option("maxFilesPerTrigger", 1).parquet(stream_dir)
scored = add_eng(sdf)
scored = model.transform(scored).select("label", "prediction")
t = time.time()
q = (scored.writeStream.format("memory").queryName("scored").outputMode("append")
     .trigger(availableNow=True).option("checkpointLocation", ck).start())
q.awaitTermination(); stream_s = time.time() - t
got = spark.table("scored")
n_scored = got.count()
stream_acc = got.filter(F.col("label") == F.col("prediction")).count() / n_scored
batches = len(q.recentProgress)
print(f"streamed rows={n_scored} in {stream_s:.1f}s -> {n_scored/stream_s:,.0f} rows/s over {batches} micro-batches; accuracy on jittered stream={stream_acc:.3f}")
RESULTS["streaming"] = dict(rows=n_scored, seconds=stream_s, rows_per_s=n_scored / stream_s, batches=batches, acc=stream_acc)

In [ ]:
with open("results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=float)
print(json.dumps(RESULTS["final"], indent=1, default=float)[:600])
spark.stop()